# Swarm Clock, Part 2d: clock (accumulation) vs recency (saturation)
Same runtime, after 2b and 2c. Pre-declared reading:
- Clock: text distance keeps rising with time lag; restricted test on lags >= 6 h stays positive.
- Recency: distance rises over minutes to hours, then plateaus; long-lag test near zero.
Calibrated against synthetic `clock` and `star` lineages (planted members only).

In [ ]:
BINS = [0, 1/6, 0.5, 1, 3, 6, 12, 24, 72, 1e9]
BIN_LABELS = ["<10m","10-30m","30m-1h","1-3h","3-6h","6-12h","12-24h","1-3d",">3d"]
def fam_pairs(sub, k=3):
    sets = [shingle_set(t, k) for t in sub.m]; n = len(sets)
    i, j = np.triu_indices(n, 1)
    d = np.array([1 - len(sets[a] & sets[b]) / max(1, len(sets[a] | sets[b])) for a, b in zip(i, j)])
    th = sub.t.astype("int64").values / 3.6e12
    lab = sub.label.values
    return {"d": d, "t": th, "i": i, "j": j, "same_label": lab[i] == lab[j],
            "strata": sub.stratum.values if "stratum" in sub else np.zeros(n)}
def lag_curve(P):
    lag = np.abs(P["t"][P["i"]] - P["t"][P["j"]])
    b = np.digitize(lag, BINS) - 1
    return np.array([P["d"][b == k].mean() if (b == k).sum() >= 5 else np.nan for k in range(len(BIN_LABELS))])
def rho_window(P, t, lo, hi):
    lag = np.abs(t[P["i"]] - t[P["j"]]); m = (lag >= lo) & (lag < hi)
    if m.sum() < 10 or np.ptp(lag[m]) == 0 or np.ptp(P["d"][m]) == 0: return np.nan
    return np.corrcoef(rankdata(P["d"][m]), rankdata(lag[m]))[0,1]
def window_test(Ps, lo, hi, B=999):
    obs_v = [rho_window(P, P["t"], lo, hi) for P in Ps]
    obs = np.nanmean(obs_v) if np.any(~np.isnan(obs_v)) else np.nan
    if np.isnan(obs): return np.nan, np.nan, obs_v
    null = []
    for _ in range(B):
        vals = []
        for P in Ps:
            t = P["t"].copy()
            for s in np.unique(P["strata"]):
                idx = np.where(P["strata"] == s)[0]; t[idx] = t[rng.permutation(idx)]
            vals.append(rho_window(P, t, lo, hi))
        if np.any(~np.isnan(vals)): null.append(np.nanmean(vals))
    null = np.array(null)
    return obs, (1 + np.sum(null >= obs)) / (len(null) + 1), obs_v

## 1. Real families (primary config and template-removed ablation)

In [ ]:
def real_pairs(D, fams):
    X = reps_with_strata(D, fams, "length")
    return [(f, fam_pairs(X[X.family == f])) for f in X.family.unique()]
REAL = {"primary": real_pairs(D_primary, elig.index), "template_removed": real_pairs(D_ab, elig_ab.index)}
rows = []
for name, FP in REAL.items():
    for f, P in FP:
        rows.append({"set": name, "family": f, **dict(zip(BIN_LABELS, np.round(lag_curve(P), 3)))})
CURVES = pd.DataFrame(rows); print(CURVES.to_string(index=False))

In [ ]:
WIN = [("short (<6h)", 0, 6), ("long (>=6h)", 6, 1e9), ("very long (>=24h)", 24, 1e9)]
WT = []
for name, FP in REAL.items():
    Ps = [P for _, P in FP]
    for wname, lo, hi in WIN:
        T, p, v = window_test(Ps, lo, hi)
        WT.append({"data": name, "window": wname, "families_with_pairs": int(np.sum(~np.isnan(v))), "T": round(T,3) if not np.isnan(T) else T,
                   "p": round(p,4) if not np.isnan(p) else p, "per_family": np.round(v, 3).tolist()})
print(pd.DataFrame(WT).to_string(index=False))

## 2. Synthetic reference curves and window tests (planted members only, same time span as real long families)

In [ ]:
def synth_pairs(kind, p, n_lin=20, days=7):
    P = plant(kind, p, n_lin=n_lin, days=days)
    out = []
    for li, sub in P.groupby("lin"):
        sub = sub.assign(stratum=0); out.append(fam_pairs(sub))
    return out
SYN = {f"{kind} p={p}": synth_pairs(kind, p) for kind in ["clock", "star"] for p in [0.02, 0.05]}
srows, swt = [], []
for name, Ps in SYN.items():
    srows.append({"set": name, **dict(zip(BIN_LABELS, np.round(np.nanmean([lag_curve(P) for P in Ps], axis=0), 3)))})
    for wname, lo, hi in WIN:
        # per-lineage tests, report rejection rate
        ps = [window_test([P], lo, hi, B=199)[1] for P in Ps]
        swt.append({"synthetic": name, "window": wname, "reject_rate": round(np.nanmean(np.array(ps) < .05), 2)})
print(pd.DataFrame(srows).to_string(index=False)); print(pd.DataFrame(swt).to_string(index=False))

## 3. Figure: distance vs time lag (real families vs synthetic references)

In [ ]:
x = np.arange(len(BIN_LABELS))
fig, ax = plt.subplots(figsize=(7,4))
for _, r in CURVES[CURVES.set == "primary"].iterrows():
    ax.plot(x, r[BIN_LABELS].astype(float).values, "-o", ms=3, lw=1, label=f"real fam {r.family}")
for _, r in pd.DataFrame(srows).iterrows():
    ax.plot(x, r[BIN_LABELS].astype(float).values, "--", lw=2, label=r.set)
ax.set_xticks(x); ax.set_xticklabels(BIN_LABELS, rotation=30); ax.set_ylabel("mean text distance (1 - Jaccard)")
ax.set_xlabel("time lag between artifacts"); ax.legend(fontsize=7, ncol=2); plt.tight_layout()
plt.savefig("fig_lag_curves.png", dpi=200); plt.show()

## 4. Same agent label vs different label at short lag (is short-range similarity cross-agent copying or one agent repeating itself?)

In [ ]:
lab_rows = []
for f, P in REAL["primary"]:
    lag = np.abs(P["t"][P["i"]] - P["t"][P["j"]]); m = lag < 1
    s, o = P["d"][m & P["same_label"]], P["d"][m & ~P["same_label"]]
    lab_rows.append({"family": f, "pairs<1h same label": len(s), "d same": round(s.mean(),3) if len(s) else np.nan,
                     "pairs<1h diff label": len(o), "d diff": round(o.mean(),3) if len(o) else np.nan})
print(pd.DataFrame(lab_rows).to_string(index=False))
CURVES.to_csv("data/lag_curves_real.csv", index=False); pd.DataFrame(WT).to_csv("data/window_tests_real.csv", index=False)
pd.DataFrame(srows).to_csv("data/lag_curves_synth.csv", index=False); pd.DataFrame(swt).to_csv("data/window_tests_synth.csv", index=False)

## 5. Cross-label window tests (exclude same-label pairs). Decisive test: is short-range signal cross-agent copying or self-revision?

In [ ]:
def rho_window_xlabel(P, t, lo, hi):
    lag = np.abs(t[P["i"]] - t[P["j"]]); m = (lag >= lo) & (lag < hi) & ~P["same_label"]
    if m.sum() < 10 or np.ptp(lag[m]) == 0 or np.ptp(P["d"][m]) == 0: return np.nan
    return np.corrcoef(rankdata(P["d"][m]), rankdata(lag[m]))[0,1]
_orig = rho_window
rho_window = rho_window_xlabel
XL = []
for name, FP in REAL.items():
    Ps = [P for _, P in FP]
    for wname, lo, hi in WIN:
        T, p, v = window_test(Ps, lo, hi)
        XL.append({"data": name, "window": wname, "T": T, "p": p, "per_family": np.round(v, 3).tolist()})
rho_window = _orig
print(pd.DataFrame(XL).to_string(index=False)); pd.DataFrame(XL).to_csv("data/window_tests_crosslabel.csv", index=False)